# p002 A5の計算精度診断（提出用ではありません）

Public 0.937成功版のA5だけを、同じ5 checkpoint・前処理・MICRO=8・GPU 0で比較します。
**T4 x2・Internet OFF** にして、新しいprivate Notebookで実行してください。
Inputsは **大会データ**、**tonylica/rsna-knee-bend-dinov3-0917-repro-assets Version 5**、
**mattiaangeli/knee-mri-fold-weights Version 2** の3つです（既存p002の5 Inputsのままでも可）。

train_series.csvの4つのメタデータ列だけを読み、seedで64検査を固定します。
train.csv・Report・gold/weakラベルは読みません。これは速度・数値安定性の診断であり、AUC/CVではありません。
同一検査を一度だけ前処理し、保存cacheのSHAを固定してBF16/FP16/FP32で各3回比較します。
読み込み・前処理・warmupと、CUDA同期したforward所要時間を別々に保存します。
BF16が環境で使えない場合は失敗として記録し、別精度へ置き換えません。

実行後はOutputの`a5-precision-probe-.../report.json`、`comparisons.csv`、raw確率/logit NPYを確認します。
`adoption_decision`は常に未決定です。小さい差でも隠しtestで同じ順位・スコアになる保証はありません。
**submission.csvは生成せず、提出・アップロード・学習も行いません。** FP16を自動採用しません。
検査数・MICRO・checkpoint・画像条件は初回比較では変更しないでください。
実MRI・公開重みによるこの診断の完走はローカルでは未検証です。


In [ ]:
import sys, types
if 'rsna_a5_precision_probe' in sys.modules:
    raise RuntimeError('Restart with a fresh Kaggle session')
probe = types.ModuleType('rsna_a5_precision_probe')
sys.modules[probe.__name__] = probe
exec(compile('"""Label-free, same-input A5 numerical/speed probe embedded by build_a5_precision_probe.py."""\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport itertools\nimport json\nimport math\nimport platform\nimport random\nimport statistics\nimport subprocess\nimport sys\nimport time\nimport warnings\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\n\ndef digest(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef select_uids(values, count, seed):\n    values = set(values)\n    if count < 2 or len(values) < count or any(not isinstance(uid, str) or not uid.strip() for uid in values):\n        raise ValueError("Need at least the requested number of distinct nonempty train UIDs (count >= 2)")\n    return sorted(values, key=lambda uid: (hashlib.sha256(f"{seed}:{uid}".encode()).hexdigest(), uid))[:count]\n\n\ndef average_ranks(values):\n    if not values or not all(math.isfinite(float(value)) for value in values):\n        raise ValueError("Ranks require finite nonempty values")\n    indices = sorted(range(len(values)), key=values.__getitem__)\n    result = [0.0] * len(values)\n    begin = 0\n    while begin < len(indices):\n        end = begin + 1\n        while end < len(indices) and values[indices[end]] == values[indices[begin]]:\n            end += 1\n        for index in indices[begin:end]:\n            result[index] = (begin + end - 1) / 2\n        begin = end\n    return result\n\n\ndef compare_vectors(reference, candidate):\n    if len(reference) != len(candidate) or len(reference) < 2:\n        raise ValueError("Need equally sized vectors with at least two studies")\n    reference, candidate = list(map(float, reference)), list(map(float, candidate))\n    rank_r, rank_c = average_ranks(reference), average_ranks(candidate)\n    diffs = [abs(a - b) for a, b in zip(reference, candidate)]\n    rank_diffs = [abs(a - b) / (len(reference) - 1) for a, b in zip(rank_r, rank_c)]\n    inversions = tie_changes = comparable = 0\n    for i, j in itertools.combinations(range(len(reference)), 2):\n        dr, dc = reference[i] - reference[j], candidate[i] - candidate[j]\n        inversions += int(dr * dc < 0)\n        tie_changes += int((dr == 0) != (dc == 0))\n        comparable += int(dr != 0 and dc != 0)\n    center = (len(reference) - 1) / 2\n    denom = math.sqrt(sum((x - center) ** 2 for x in rank_r) * sum((x - center) ** 2 for x in rank_c))\n    return {\n        "max_abs_probability_difference": max(diffs),\n        "mean_abs_probability_difference": statistics.mean(diffs),\n        "max_normalized_rank_difference": max(rank_diffs),\n        "mean_normalized_rank_difference": statistics.mean(rank_diffs),\n        "rank_spearman": sum((a - center) * (b - center) for a, b in zip(rank_r, rank_c)) / denom if denom else None,\n        "pair_inversions": inversions,\n        "pair_tie_changes": tie_changes,\n        "comparable_pairs": comparable,\n    }\n\n\ndef check_probability_shape(array, expected):\n    import numpy as np\n\n    if array.shape != expected:\n        raise ValueError(f"Unexpected output shape {array.shape}, expected {expected}")\n    bad = int((~np.isfinite(array)).sum())\n    outside = int(((array < 0) | (array > 1)).sum())\n    return {"nonfinite": bad, "outside_probability_range": outside, "valid": bad == outside == 0}\n\n\ndef resolve_inputs(contract):\n    resolved, receipts = {}, []\n    for entry in contract["inputs"]:\n        roots = sorted({Path(root).resolve() for root in entry["roots"] if Path(root).is_dir()})\n        if len(roots) != 1:\n            raise ValueError(f"Exactly one fixed mount required for {entry[\'key\']}: {roots}")\n        root = roots[0]\n        resolved[entry["key"]] = root\n        for expected in entry["files"]:\n            path = root / expected["path"]\n            if not path.is_file() or path.stat().st_size != expected["bytes"] or digest(path) != expected["sha256"]:\n                raise ValueError(f"Fixed input hash/size mismatch: {path}")\n            receipts.append({"path": str(path), "bytes": expected["bytes"], "sha256": expected["sha256"]})\n    return resolved, receipts\n\n\ndef environment(torch):\n    versions = {}\n    for package in ("torch", "torchvision", "timm", "numpy", "pandas", "pydicom", "opencv-python-headless"):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            versions[package] = None\n    return {\n        "python": sys.version,\n        "platform": platform.platform(),\n        "packages": versions,\n        "cuda": torch.version.cuda,\n        "cudnn": torch.backends.cudnn.version(),\n        "gpus": [\n            {\n                "name": torch.cuda.get_device_name(i),\n                "capability": list(torch.cuda.get_device_capability(i)),\n                "total_memory": torch.cuda.get_device_properties(i).total_memory,\n                "native_bf16_hardware": torch.cuda.get_device_capability(i) >= (8, 0),\n            }\n            for i in range(torch.cuda.device_count())\n        ],\n        "tf32_matmul": torch.backends.cuda.matmul.allow_tf32,\n        "tf32_cudnn": torch.backends.cudnn.allow_tf32,\n        "cudnn_benchmark": torch.backends.cudnn.benchmark,\n        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),\n    }\n\n\ndef save_report(path, report):\n    path.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef make_models(ns, checkpoint_paths, torch):\n    models, configs, folds = [], [], []\n    started = time.perf_counter()\n    keys = ("backbone", "img", "cond", "pool", "stem", "n_slice", "n_meta", "norm")\n    for checkpoint in checkpoint_paths:\n        # The full checkpoint has already passed the pinned SHA-256 gate.\n        payload = torch.load(checkpoint, map_location="cpu", weights_only=False)\n        cfg = payload["cfg"]\n        if cfg.get("n_meta", 0) != 0 or cfg.get("n_slice", 16) != 16 or cfg["img"] != 336:\n            raise ValueError("A5 checkpoint changed metadata/slice/image input contract")\n        if configs and any(cfg.get(key) != configs[0].get(key) for key in keys):\n            raise ValueError("A5 folds disagree on model/preprocessing config")\n        stem = cfg.get("stem", "native")\n        channels = 3 if stem == "compress" else cfg.get("n_slice", 16)\n        kwargs = {"img_size": cfg["img"]} if "vit_" in cfg["backbone"] else {}\n        encoder = ns["timm"].create_model(cfg["backbone"], pretrained=False, num_classes=0, in_chans=channels, **kwargs)\n        if cfg["cond"] == "token":\n            encoder = ns["ViTSlotToken"](encoder, ns["N_SLOT_TYPES"])\n        model = ns["Net"](\n            encoder, cfg["cond"], cfg.get("n_meta", 0), cfg["pool"], stem=stem, n_slice=cfg.get("n_slice", 16)\n        )\n        model.load_state_dict(payload["state_dict"], strict=True)\n        models.append(model.eval().to("cuda:0"))\n        folds.append(int(payload["fold"]))\n        configs.append(cfg)\n        del payload\n    if folds != list(range(5)):\n        raise ValueError(f"Expected ordered folds 0..4, got {folds}")\n    torch.cuda.synchronize(0)\n    ns.update(models=models, CFG=configs[-1], DEV="cuda:0")\n    return {"seconds": time.perf_counter() - started, "folds": folds, "configs": configs}\n\n\ndef prepare_train_cache(ns, competition, run_dir, count, seed, np, pd):\n    started = time.perf_counter()\n    cols = ["StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane", "Fat_Suppression"]\n    metadata = competition / "train_series.csv"\n    frame = pd.read_csv(metadata, usecols=cols, dtype={cols[0]: str, cols[1]: str})\n    ids = select_uids(frame.StudyInstanceUID.unique().tolist(), count, seed)\n    selected = frame[frame.StudyInstanceUID.isin(ids)]\n    by = {uid: group.to_dict("records") for uid, group in selected.groupby("StudyInstanceUID")}\n    (run_dir / "uids.txt").write_text("\\n".join(ids) + "\\n", encoding="utf-8")\n    selected.to_csv(run_dir / "selected_series_metadata.csv", index=False)\n    ns["SERIES_ROOT"] = competition / "train_series"\n    if not ns["SERIES_ROOT"].is_dir():\n        raise FileNotFoundError(ns["SERIES_ROOT"])\n    image_path = run_dir / "preprocessed_images.npy"\n    mask_path = run_dir / "preprocessed_masks.npy"\n    images = np.lib.format.open_memmap(image_path, mode="w+", dtype="uint8", shape=(count, 6, 16, 336, 336))\n    masks = np.lib.format.open_memmap(mask_path, mode="w+", dtype="uint8", shape=(count, 6))\n    per_study = []\n    decode_failures = [0]\n    original_read_crop = ns["read_crop"]\n\n    def observed_read_crop(path):\n        result = original_read_crop(path)\n        if result is None:\n            decode_failures[0] += 1\n        return result\n\n    ns["read_crop"] = observed_read_crop\n    try:\n        for index, uid in enumerate(ids):\n            before = decode_failures[0]\n            _, image, mask = ns["build_study"]((index, uid, by[uid]))\n            if not (mask > 0).any() or not image.any():\n                raise ValueError(f"No valid A5 image slots for fixed train study {uid}")\n            images[index], masks[index] = image, mask\n            per_study.append(\n                {\n                    "uid": uid,\n                    "mask_counts": mask.tolist(),\n                    "nonzero_slices": (image.max(axis=(2, 3)) > 0).sum(axis=1).tolist(),\n                    "failed_crop_reads": decode_failures[0] - before,\n                }\n            )\n            print(f"A5 fixed train preprocessing {index + 1}/{count}", flush=True)\n        images.flush()\n        masks.flush()\n    finally:\n        ns["read_crop"] = original_read_crop\n    del images, masks\n    images = np.load(image_path, mmap_mode="c", allow_pickle=False)\n    masks = np.load(mask_path, mmap_mode="c", allow_pickle=False)\n    return (\n        ids,\n        images,\n        masks,\n        {\n            "seconds": time.perf_counter() - started,\n            "selection": "sha256(seed:UID) ascending; train_series only",\n            "metadata_sha256": digest(metadata),\n            "metadata_columns_read": cols,\n            "uid_sha256": digest(run_dir / "uids.txt"),\n            "images_sha256": digest(image_path),\n            "masks_sha256": digest(mask_path),\n            "image_shape": list(images.shape),\n            "per_study": per_study,\n            "failed_crop_reads": decode_failures[0],\n            "source_preprocessing_preserved": True,\n            "execution": "serial once; forward timing excludes preprocessing",\n        },\n    )\n\n\ndef measure_once(ns, images, masks, mode, torch, trace_dtypes=False):\n    dtype, enabled = {"bf16": (torch.bfloat16, True), "fp16": (torch.float16, True), "fp32": (torch.float32, False)}[\n        mode\n    ]\n    ns["AMP_DT"], ns["AMP_ON"] = dtype, enabled\n    # Same input/model/MICRO/device and autocast scope as p002 _micro().\n    observed, hooks = {}, []\n    logits_by_fold = [[] for _ in ns["models"]]\n\n    def capture_logits(fold):\n        def hook(module, inputs, output):\n            # Retain without modifying logits; reduction/copy happen after synchronized timing.\n            logits_by_fold[fold].append(output.detach())\n\n        return hook\n\n    for fold, model in enumerate(ns["models"]):\n        hooks.append(model.register_forward_hook(capture_logits(fold)))\n    if trace_dtypes:\n\n        def capture(name):\n            def hook(module, inputs, output):\n                tensor = output[0] if isinstance(output, (tuple, list)) else output\n                observed[name] = {\n                    "output_dtype": str(tensor.dtype) if hasattr(tensor, "dtype") else type(tensor).__name__,\n                    "cuda_autocast_enabled": torch.is_autocast_enabled("cuda"),\n                    "cuda_autocast_dtype": str(torch.get_autocast_dtype("cuda")),\n                }\n\n            return hook\n\n        for fold, model in enumerate(ns["models"]):\n            # The first Linear/Conv2d and final logits expose actual dtypes, not just requested AMP.\n            for name, module in model.named_modules():\n                if isinstance(module, (torch.nn.Linear, torch.nn.Conv2d)):\n                    hooks.append(module.register_forward_hook(capture(f"fold{fold}.{name}")))\n                    break\n            hooks.append(model.register_forward_hook(capture(f"fold{fold}.logits")))\n    torch.cuda.synchronize(0)\n    torch.cuda.reset_peak_memory_stats(0)\n    start = time.perf_counter()\n    try:\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter("always")\n            prediction = ns["predict"](images, masks)\n            torch.cuda.synchronize(0)\n            seconds = time.perf_counter() - start\n    finally:\n        for hook in hooks:\n            hook.remove()\n    logits = torch.stack([torch.cat(values, dim=0).float() for values in logits_by_fold]).cpu().numpy()\n    if logits.shape != prediction.shape:\n        raise ValueError("Captured raw logits do not cover every fold/study/label")\n    return (\n        prediction,\n        logits,\n        {\n            "seconds": seconds,\n            "peak_allocated_bytes": torch.cuda.max_memory_allocated(0),\n            "peak_reserved_bytes": torch.cuda.max_memory_reserved(0),\n            "warnings": sorted({str(item.message) for item in caught}),\n            "dtype": str(dtype),\n            "autocast": enabled,\n            "observed_dtypes": observed,\n            "nonfinite_logits": int((~ns["np"].isfinite(logits)).sum()),\n            "timing_observer": "nonmutating forward hooks retain logits; checks/copy/save excluded",\n        },\n    )\n\n\ndef comparisons(raw, labels, pd, np):\n    rows = []\n    for left, right in itertools.combinations(sorted(raw), 2):\n        # compare against FP32 where present and BF16 otherwise.\n        reference, candidate = (right, left) if right == "fp32" else (left, right)\n        for fold in range(5):\n            for col, label in enumerate(labels):\n                rows.append(\n                    {\n                        "reference": reference,\n                        "candidate": candidate,\n                        "fold": fold,\n                        "label": label,\n                        **compare_vectors(raw[reference][fold, :, col], raw[candidate][fold, :, col]),\n                    }\n                )\n        # Reproduce the source\'s ordinal rank followed by fold mean, including its tie handling.\n        rank_means = {}\n        for mode in (reference, candidate):\n            per_fold = raw[mode].argsort(axis=1).argsort(axis=1).astype(np.float64)\n            rank_means[mode] = (per_fold / max(raw[mode].shape[1] - 1, 1)).mean(axis=0)\n        for col, label in enumerate(labels):\n            rows.append(\n                {\n                    "reference": reference,\n                    "candidate": candidate,\n                    "fold": "p002_ordinal_rank_mean",\n                    "label": label,\n                    **compare_vectors(rank_means[reference][:, col], rank_means[candidate][:, col]),\n                }\n            )\n    return pd.DataFrame(rows)\n\n\ndef run_probe(contract, definition_source, studies=64, seed=20261005, repeats=3):\n    """Called only by the user on Kaggle. Never creates or modifies a submission CSV."""\n    if not Path("/kaggle/working").is_dir() or not Path("/kaggle/input").is_dir():\n        raise RuntimeError("This diagnostic is restricted to Kaggle; no local MRI/model execution")\n    if studies != 64 or repeats < 2:\n        raise ValueError("First comparison uses exactly 64 fixed train studies and at least two repeats")\n    if hashlib.sha256(definition_source.encode()).hexdigest() != contract["source"]["definitions_sha256"]:\n        raise ValueError("Definition source hash mismatch")\n    roots, input_receipts = resolve_inputs(contract)\n    wheel = roots["offline_timm"] / "timm-1.0.22-py3-none-any.whl"\n    if "timm" in sys.modules:\n        raise RuntimeError("Use a fresh session before installing the fixed offline timm wheel")\n    subprocess.run(\n        [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--force-reinstall", str(wheel)], check=True\n    )\n    import numpy as np\n    import pandas as pd\n    import torch\n\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:\n        raise RuntimeError("Use T4 x2, as in the scored p002 notebook")\n    if any("T4" not in torch.cuda.get_device_name(i) for i in range(2)):\n        raise RuntimeError("This comparison must use T4 x2")\n    if importlib.metadata.version("timm") != "1.0.22":\n        raise RuntimeError("Offline timm version mismatch")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    run_dir = Path("/kaggle/working") / (\n        "a5-precision-probe-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")\n    )\n    run_dir.mkdir(exist_ok=False)\n    report_path = run_dir / "report.json"\n    report = {\n        "schema": contract["schema"],\n        "status": "running",\n        "run_directory": str(run_dir),\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "seed": seed,\n        "studies": studies,\n        "repeats": repeats,\n        "micro": 8,\n        "folds": list(range(5)),\n        "device": "cuda:0",\n        "labels_read": False,\n        "gold_used": False,\n        "training_performed": False,\n        "public_score_measured": False,\n        "submission_generated": False,\n        "adoption_decision": "undecided; diagnostic only; manual review required; no automatic precision change",\n        "environment": environment(torch),\n        "input_files": input_receipts,\n        "source": contract["source"],\n        "modes": {},\n        "comparisons": {},\n        "runtime_scope": "A5 only; full 9-hour submission budget remains unverified",\n    }\n    save_report(report_path, report)\n    try:\n        ns = {"__name__": "a5_p002_definitions"}\n        exec(compile(definition_source, "<p002-a5-definitions>", "exec"), ns)\n        ns["cv2"].setNumThreads(1)\n        paths = sorted((roots["consolidated"] / "knee-mri-fold-weights").glob("*_f*.pt"))\n        expected_paths = sorted(\n            roots["consolidated"] / item["path"]\n            for entry in contract["inputs"]\n            if entry["key"] == "consolidated"\n            for item in entry["files"]\n        )\n        if paths != expected_paths or len(paths) != 5:\n            raise ValueError("Expected exactly the five hash-verified A5 checkpoints")\n        report["model_load"] = make_models(ns, paths, torch)\n        ids, images, masks, cache = prepare_train_cache(ns, roots["competition"], run_dir, studies, seed, np, pd)\n        report["preprocessing"] = cache\n        report["label_order"] = list(ns["LABELS"])\n        save_report(report_path, report)\n        modes = list(contract["modes"])\n        active = []\n        for mode in modes:\n            record = {"status": "warming_up", "measurements": []}\n            report["modes"][mode] = record\n            try:\n                prediction, _, timing = measure_once(ns, images[:8], masks[:8], mode, torch, trace_dtypes=True)\n                valid = check_probability_shape(prediction, (5, 8, 12))\n                record["warmup"] = {**timing, **valid}\n                if not valid["valid"] or timing["nonfinite_logits"]:\n                    raise ValueError("Warmup produced invalid probabilities/logits")\n                record["status"] = "ready"\n                active.append(mode)\n            except Exception as exc:\n                record.update(status="failed", failure=f"{type(exc).__name__}: {exc}")\n                gc.collect()\n                torch.cuda.empty_cache()\n            save_report(report_path, report)\n        raw = {}\n        for repeat in range(repeats):\n            # Rotate mode order; do not silently change dtype if a mode fails.\n            order = modes[repeat % len(modes) :] + modes[: repeat % len(modes)]\n            for mode in order:\n                if mode not in active:\n                    continue\n                record = report["modes"][mode]\n                try:\n                    prediction, logits, timing = measure_once(ns, images, masks, mode, torch)\n                    valid = check_probability_shape(prediction, (5, studies, 12))\n                    raw_path = run_dir / f"raw_probabilities_{mode}_repeat{repeat + 1}.npy"\n                    np.save(raw_path, prediction, allow_pickle=False)\n                    logits_path = run_dir / f"raw_logits_{mode}_repeat{repeat + 1}.npy"\n                    np.save(logits_path, logits, allow_pickle=False)\n                    record["measurements"].append(\n                        {\n                            "repeat": repeat + 1,\n                            **timing,\n                            **valid,\n                            "raw_path": raw_path.name,\n                            "sha256": digest(raw_path),\n                            "raw_logits_path": logits_path.name,\n                            "raw_logits_sha256": digest(logits_path),\n                        }\n                    )\n                    if not valid["valid"] or timing["nonfinite_logits"]:\n                        raise ValueError("Invalid probabilities/logits saved; no neutralization permitted")\n                    if mode in raw:\n                        record["measurements"][-1]["max_abs_repeat_difference"] = float(\n                            np.abs(raw[mode] - prediction).max()\n                        )\n                    else:\n                        raw[mode] = prediction.copy()\n                        for fold in range(5):\n                            frame = pd.DataFrame(prediction[fold], columns=ns["LABELS"])\n                            frame.insert(0, "StudyInstanceUID", ids)\n                            frame.to_csv(run_dir / f"a5_raw_{mode}_fold{fold}.csv", index=False)\n                    print(f"A5 {mode} repeat {repeat + 1}/{repeats}: {timing[\'seconds\']:.2f} sec", flush=True)\n                except Exception as exc:\n                    record.update(status="failed", failure=f"{type(exc).__name__}: {exc}")\n                    active.remove(mode)\n                    raw.pop(mode, None)\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                save_report(report_path, report)\n        for mode in active:\n            record = report["modes"][mode]\n            seconds = [item["seconds"] for item in record["measurements"]]\n            record.update(\n                status="completed",\n                median_seconds=statistics.median(seconds),\n                min_seconds=min(seconds),\n                max_seconds=max(seconds),\n                studies_per_second=studies / statistics.median(seconds),\n            )\n        table = comparisons(raw, ns["LABELS"], pd, np)\n        table.to_csv(run_dir / "comparisons.csv", index=False)\n        report["comparisons"] = {\n            "path": "comparisons.csv",\n            "rows": len(table),\n            "sha256": digest(run_dir / "comparisons.csv"),\n            "raw_reference_repeat": 1,\n            "rank_scope": "64 fixed train studies; not the full hidden test",\n            "accuracy_claim": "none; no labels read and no precision adopted",\n        }\n        # Verify that every dtype saw the identical cached bytes.\n        if (\n            digest(run_dir / "preprocessed_images.npy") != cache["images_sha256"]\n            or digest(run_dir / "preprocessed_masks.npy") != cache["masks_sha256"]\n        ):\n            raise ValueError("Preprocessed cache changed during precision comparison")\n        report["status"] = "diagnostic_complete" if len(active) == len(modes) else "diagnostic_incomplete"\n        report["preprocessing"]["cache_hash_unchanged"] = True\n    except Exception as exc:\n        report.update(status="failed", failure=f"{type(exc).__name__}: {exc}")\n        save_report(report_path, report)\n        raise\n    save_report(report_path, report)\n    return report\n', '<a5-probe-runtime>', 'exec'), probe.__dict__)
CONTRACT = {'schema': 'p002_a5_precision_probe_v1', 'inputs': [{'key': 'consolidated', 'ref': 'tonylica/rsna-knee-bend-dinov3-0917-repro-assets', 'version': 5, 'roots': ['/kaggle/input/rsna-knee-bend-dinov3-0917-repro-assets', '/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets'], 'files': [{'path': 'knee-mri-fold-weights/m_f0.pt', 'bytes': 94000039, 'sha256': 'b447a665a92ed65e0f4cacf6800940ba7a6d585d0c36154ce18dee60ec48d5cd'}, {'path': 'knee-mri-fold-weights/m_f1.pt', 'bytes': 94000039, 'sha256': '30bbe5f9afa24af57fd69bad293a4b9fe3ca2e1d737381bc06e469c5ed380e76'}, {'path': 'knee-mri-fold-weights/m_f2.pt', 'bytes': 94000039, 'sha256': '5bdc4f72471e7c5db01f2f919ba39212087cbb5f2f76b619e4ba38446800b782'}, {'path': 'knee-mri-fold-weights/m_f3.pt', 'bytes': 94000039, 'sha256': 'f757ddc9531f903c6cd0729cadc7051fcdca03b281c89c39cc7957ab44fc2243'}, {'path': 'knee-mri-fold-weights/m_f4.pt', 'bytes': 94000039, 'sha256': '066490a5c17b3887d1bfda1427f45b14cab416e7b1dcc16afc9a003cdcc08aad'}]}, {'key': 'offline_timm', 'ref': 'mattiaangeli/knee-mri-fold-weights', 'version': 2, 'roots': ['/kaggle/input/knee-mri-fold-weights', '/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights'], 'files': [{'path': 'timm-1.0.22-py3-none-any.whl', 'bytes': 2530238, 'sha256': '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0'}]}, {'key': 'competition', 'ref': 'rsna-knee-abnormality-detection', 'version': None, 'roots': ['/kaggle/input/rsna-knee-abnormality-detection', '/kaggle/input/competitions/rsna-knee-abnormality-detection'], 'files': []}], 'source': {'notebook_sha256': 'ba9491ac1e214ba55ca181de55118c9a793fe7f6163fc221fe1e68d596259dca', 'cell_index': 5, 'cell_sha256': '8f35d00089e7d2e0a1cafe00c1d8909ad3c1c864ac8408c27a3e7b0c8e2f08f0', 'segments': [{'start': 2, 'end': 2, 'sha256': '787e1b6c5e8b44d63af9667c80025341f9bfaf270e9eaec5a23d11a0be812b8d'}, {'start': 3, 'end': 3, 'sha256': '890fd0e822a71c5bee9c0477b9c7169a7bf1b0fcb4f8bf66cd322b24d397e900'}, {'start': 4, 'end': 4, 'sha256': '3cb9ffccba5146b2dd177a30c3057a4395ebd5c745d0789d2ed95d0cf4afe271'}, {'start': 5, 'end': 5, 'sha256': 'e8005793a7c28c1d6abe8ca70408dd1610512d6bc6c7990c7a0ba525b9ae0bbd'}, {'start': 6, 'end': 6, 'sha256': 'd34aea06c990aeedd2f8f5ff809c1180b92fccca0381269b7c18654043b9a374'}, {'start': 7, 'end': 7, 'sha256': 'd7a11d5edc0f0628b4a8d665a78db6dd1062d88065d316b4293c450edf76030b'}, {'start': 8, 'end': 8, 'sha256': '6d42ba55b2c7b47ed194b810f897793cda85b0eb72e6937749fdd8276ee9fb93'}, {'start': 9, 'end': 9, 'sha256': '1c69c8907ebc3cfe17c571c2697c79f02555929ef119ebca55883a4b7cba359f'}, {'start': 10, 'end': 10, 'sha256': 'b91a3bc5e2064d025a8ae122555c96948ec5b4895d53d809c4b66b71a4915133'}, {'start': 11, 'end': 11, 'sha256': '1e9ac7e82a003795131afd16330ff3802df567159ee50117eed9616cddd500f5'}, {'start': 12, 'end': 12, 'sha256': '766fdc6e4d19f35fb51ac36854ee42bb402c9067a8ca883e31f21e67c41a6c7f'}, {'start': 15, 'end': 15, 'sha256': '388749977d43bc8124c3499043c6c2018e0cc5555ca636e5566df232b9f4bb85'}, {'start': 16, 'end': 16, 'sha256': '6b5df77feef89e72a82a03db2e9c6c0fa65a1b6b1f0669c9c6036d49f4308c34'}, {'start': 17, 'end': 17, 'sha256': 'd7f34a5d37f6645b5e47f46ddc5e06e4589e4d66226a4431cc76bb6a1b99307d'}, {'start': 18, 'end': 18, 'sha256': 'ea7352f48e670cb6093ef0218b023e6d604fe5ffd19813edace28c7ecb44880a'}, {'start': 19, 'end': 19, 'sha256': '84aeb531cbd7a805f52b41e0834018f101c6c16f09311da710f4e1c19dea6b5e'}, {'start': 20, 'end': 20, 'sha256': 'b9e91ea429d3b9b0231198439f67d0623d3c304535242455d106bba845a912d1'}, {'start': 21, 'end': 21, 'sha256': '3c4ee9ece50b1108ebc37af7cfabce3ab43ad3c2d0b3796d2801a7960d1e6f0b'}, {'start': 22, 'end': 22, 'sha256': 'c31737e1545aa0737e928b74c16bc508e294c63c3d9d4907e03a0e2d69a5bcb4'}, {'start': 42, 'end': 52, 'sha256': 'd7db95ed6f1b183a8d9fbc89a948bf5b9549ad099932c1ef677bbbea89586234'}, {'start': 54, 'end': 58, 'sha256': 'd9d18dbc1b4ed73ab83e520553bd618caabd51b1950b12ba432df41ad850a040'}, {'start': 60, 'end': 75, 'sha256': '6324090f307de5e798484d3a1f7d93ef08272be56a89972ecf5ad25bf4695566'}, {'start': 77, 'end': 80, 'sha256': '0ebd1c34930096cc1550c88eff54cebd4eaf178fb235a22c421c79be3aaa9297'}, {'start': 82, 'end': 87, 'sha256': '8dd8a6fc1a7418771de7735e2ab54b361679efe953234d7e2d0b2345d917d8c8'}, {'start': 89, 'end': 131, 'sha256': '64c1fc34abedbb381bc525718c075c5d15468700e694414d6ef1fc10913ea086'}, {'start': 140, 'end': 140, 'sha256': '431609972ee84dd8ea106056d0bccab787bbe75a42ec00fabbf4108418f3a45c'}, {'start': 142, 'end': 149, 'sha256': 'ef6049509aa00297c31f4674428a9aa4b23130de5de8c9c099e00670c3ca73ec'}, {'start': 151, 'end': 160, 'sha256': 'dd48965400419862946338544478fa1d237e68d422ed67dacb8b7cefce68b01b'}, {'start': 162, 'end': 178, 'sha256': 'ef606e06c53ed812e08242d31b97ee8cb10d47e1770ee38dbda2b65d4dcbc944'}, {'start': 180, 'end': 210, 'sha256': '7b59513ac96a477e701d311f2351ea09af82a0249774251f18ae8b3b471e6f99'}, {'start': 212, 'end': 257, 'sha256': '328daafcb4707ab3fce040462cadc51af12df1b097cc13fb90aa6a8baa513013'}, {'start': 258, 'end': 258, 'sha256': '7834228d4403fb6c1cac7f0f7d76403b7cc877d21060997e99f7105ed0be7500'}, {'start': 259, 'end': 259, 'sha256': '6dcb6990a45ababd44bfe6e9ef3cc7aebf70731c9653a6b4941b429ad3ae7a97'}, {'start': 261, 'end': 274, 'sha256': '232a9293149b252f3843611e8b47abbad5d2aa40a37f4f711b001e05d5d49ea2'}, {'start': 276, 'end': 295, 'sha256': 'c867f3df92a44635473ee8a5dfc1180983514e97b735f7ea5145dbfaa6727a25'}, {'start': 296, 'end': 296, 'sha256': 'b94bfc902747f7479b5d8754a52f9f1bed4fcf2e7848e6c64d7a674537157f34'}, {'start': 297, 'end': 297, 'sha256': '530778db3a38af59ffa39dd5762283d696e041ba8983d7a6044fc0c23c8c7c0c'}, {'start': 298, 'end': 298, 'sha256': '062b8915b045c1c80207a17aa48251d438bda63bf8408a61b4f2248e5be848d1'}, {'start': 300, 'end': 347, 'sha256': 'c0d56434926fd00c0b4bb79ea8affe6f59c27cf30adf6ee5e08dd9450d263fde'}, {'start': 349, 'end': 356, 'sha256': 'f5c50ea7dee3c7801da378b284b3177f0b7885ff6ea5c474baf3c4085ad0a603'}, {'start': 358, 'end': 368, 'sha256': '0b1d4ed53542329a316ed29ad2af7b5c0858a79eb19e2a4afaa52d1a6cec16c5'}, {'start': 370, 'end': 386, 'sha256': '2512e12b466470a5164e65b1056139a3f3f56718ee58a97fbe9797332c9401d3'}, {'start': 388, 'end': 397, 'sha256': '3592e5c72d5dbe50abb6d11994f810d6130b56e951cd5b5f605cef08af6ffef2'}, {'start': 399, 'end': 408, 'sha256': 'b44eabc9442fab0385dad1c7129a2bd37bfdc4832d84bca1dfbf8d140ed4cc0b'}, {'start': 410, 'end': 417, 'sha256': 'd5574381c39347aed307351872b667d7bc84c2adc8aa59a6e87bb6e62ae471cc'}, {'start': 419, 'end': 452, 'sha256': '3676c2ecf35afb0bfd0736d76e0a05b55903dc5894bd755aabb971226287a5f8'}, {'start': 454, 'end': 488, 'sha256': 'ffa2506f1271274bf7ef97fc76e31a1135c815a6895af0e42a38c1d9962017ca'}, {'start': 509, 'end': 519, 'sha256': '96143cd4288bb107456ca61f1ce0ac46d9142da21bcb0da69cc0cd5bf5fccd24'}, {'start': 523, 'end': 523, 'sha256': 'e4ed720eda8214198930f04a265595bad3447c5cbdd5e75d36ca54678af11f3a'}, {'start': 527, 'end': 538, 'sha256': 'c75dee43173168c07944c20b11dfc23ac9db7470bea845392a49795a63d111ab'}, {'start': 540, 'end': 568, 'sha256': '1aef01fb14f2c58fc09c8c77323f80bbb57868386c88d2d52d99ade884374ad3'}, {'start': 570, 'end': 575, 'sha256': '65e8b791bdbae2f4a34333d41c600356bd6cfa4c002a2dd1bfc73aff982d14ab'}], 'definitions_sha256': '91e70d269db9009086d9bf92cd636b00bda80da8defac9d24d56723b4a2775ef'}, 'checkpoint_count': 5, 'modes': ['bf16', 'fp16', 'fp32']}
DEFINITION_SOURCE = '# ruff: noqa\n# fmt: off\n"""Exact definitions from romantamrazov/rsna-knee-dinosaur-v4 Version 32, cell 5.\nSource notebook SHA-256: ba9491ac1e214ba55ca181de55118c9a793fe7f6163fc221fe1e68d596259dca\nUsed by the A5 diagnostic only; top-level I/O, loading, inference and blending are removed.\nPublic code attribution: https://www.kaggle.com/code/romantamrazov/rsna-knee-dinosaur-v4\n"""\n\nimport gc, os, time, warnings\n\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\n\nfrom pathlib import Path\n\nimport cv2\n\nimport numpy as np\n\nimport pandas as pd\n\nimport pydicom\n\nimport timm\n\nimport torch\n\nimport torch.nn as nn\n\nimport torch.nn.functional as F\n\nCROP_MM = 130.0\n\nSIZE = 336\n\nSLICE_BAND = (0.12, 0.88)\n\nN_SLICE = 16\n\nINTENSITY = \'slice\'\n\nSLOTS = [(\'Sagittal\', 1), (\'Sagittal\', 0), (\'Coronal\', 1), (\'Coronal\', 0), (\'Axial\', 1), (\'Axial\', 0)]\n\nN_SLOT = len(SLOTS)\n\nLABELS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\ndef ordered_files(sdir, cap=64):\n    keyed = []\n    for f in sdir.glob(\'*.dcm\'):\n        try:\n            ds = pydicom.dcmread(str(f), stop_before_pixels=True)\n            keyed.append((int(ds.InstanceNumber), str(f)))\n        except Exception:\n            continue\n        if len(keyed) >= cap * 4:\n            break\n    return [f for _, f in sorted(keyed)]\n\ndef series_side(path):\n    try:\n        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])\n    except Exception:\n        return 0.0\n\ndef read_crop(path):\n    try:\n        ds = pydicom.dcmread(path)\n        arr = ds.pixel_array.astype(np.float32)\n    except Exception:\n        return None\n    try:\n        ps = float(ds.PixelSpacing[0])\n    except Exception:\n        ps = CROP_MM / max(arr.shape)\n    half = int(round(CROP_MM / ps / 2))\n    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)\n    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))\n    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))\n    crop = arr[y0:y1, x0:x1]\n    return None if crop.size == 0 else crop\n\ndef window(crop, lo, hi, flip):\n    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)\n    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)\n    return img[:, ::-1].copy() if flip else img\n\ndef render(path, flip):\n    crop = read_crop(path)\n    if crop is None:\n        return None\n    lo, hi = np.percentile(crop[::4, ::4], [1, 99])\n    return window(crop, lo, hi, flip)\n\ndef build_study(args):\n    idx, study, recs = args\n    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)\n    mask = np.zeros(N_SLOT, np.uint8)\n    rows = pd.DataFrame(recs)\n    if len(rows):\n        for s_i, (plane, fs) in enumerate(SLOTS):\n            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]\n            if sub.empty:\n                continue\n            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)\n            if not files:\n                continue\n            flip = plane != \'Sagittal\' and series_side(files[0]) < 0\n            lo, hi = SLICE_BAND\n            i0 = int(round(lo * (len(files) - 1)))\n            i1 = int(round(hi * (len(files) - 1)))\n            avail = list(range(i0, i1 + 1))\n            if len(avail) >= N_SLICE:\n                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]\n                off = 0\n            else:\n                picks, off = (avail, (N_SLICE - len(avail)) // 2)\n            if INTENSITY == \'series\':\n                crops = [read_crop(files[p]) for p in picks]\n                got = [x for x in crops if x is not None]\n                if got:\n                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])\n                    lo_, hi_ = np.percentile(samp, [1, 99])\n                    for c, x in enumerate(crops):\n                        if x is None:\n                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])\n                        if x is not None:\n                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)\n            else:\n                for c, p in enumerate(picks):\n                    img = render(files[p], flip)\n                    if img is None:\n                        img = render(files[min(len(files) - 1, p + 1)], flip)\n                    if img is not None:\n                        out[s_i, off + c] = (img * 255).astype(np.uint8)\n            mask[s_i] = len(picks)\n    return (idx, out, mask)\n\nN_SLOT_TYPES, MASK_IDX = (6, 0)\n\ndef segment_softmax(scores, sidx, B):\n    T, K = scores.shape\n    idx = sidx.unsqueeze(1).expand(-1, K)\n    m = torch.full((B, K), float(\'-inf\'), device=scores.device, dtype=scores.dtype)\n    m = m.scatter_reduce(0, idx, scores, reduce=\'amax\', include_self=True)\n    e = (scores - m[sidx]).exp()\n    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)\n    return e / s[sidx].clamp(min=1e-06)\n\nclass MeanMaxPool(nn.Module):\n\n    def forward(self, f, sidx, B, slot=None, return_attn=False):\n        D = f.shape[1]\n        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))\n        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)\n        mean = mean / cnt.clamp(min=1).unsqueeze(1)\n        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)\n        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce=\'amax\', include_self=True)\n        return (torch.cat([mean, mx], 1), None)\n\nclass LabelAttentionPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):\n        super().__init__()\n        self.d, self.k, self.h = (d, n_labels, n_heads)\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))\n        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None\n\n    def forward(self, f, sidx, B, slot=None, return_attn=False):\n        scores = self.key(f) @ self.q.t() / self.d ** 0.5\n        if self.slot_bias is not None and slot is not None:\n            scores = scores + self.slot_bias.t()[slot]\n        a = segment_softmax(scores, sidx, B)\n        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)\n        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))\n        return (out, a)\n\nclass TokenXAttnPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):\n        super().__init__()\n        self.d, self.k = (d, n_labels)\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)\n        self.kv_norm = nn.LayerNorm(d)\n        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)\n\n    def forward(self, tok, sidx, B, slot=None, return_attn=False):\n        T, N, D = tok.shape\n        cnt = torch.bincount(sidx, minlength=B)\n        S = int(cnt.max().item())\n        starts = torch.cumsum(cnt, 0) - cnt\n        pos = torch.arange(T, device=tok.device) - starts[sidx]\n        kv = tok + self.slot_emb(slot).unsqueeze(1)\n        pad = tok.new_zeros(B, S, N, D)\n        pad[sidx, pos] = kv\n        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)\n        keep[sidx, pos] = True\n        kpm = ~keep.repeat_interleave(N, dim=1)\n        pad = self.kv_norm(pad.reshape(B, S * N, D))\n        q = self.q.unsqueeze(0).expand(B, -1, -1)\n        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)\n        cls = tok[:, 0]\n        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)\n        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)\n        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce=\'amax\', include_self=True)\n        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)\n        return (torch.cat([att, base], -1), w)\n\nclass ViTSlotToken(nn.Module):\n\n    def __init__(self, vit, n_cat, dim=None):\n        super().__init__()\n        self.vit = vit\n        d = dim or vit.embed_dim\n        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)\n        self.num_features = vit.num_features\n        self._orig_prefix = getattr(vit, \'num_prefix_tokens\', 1)\n        vit.num_prefix_tokens = self._orig_prefix + 1\n        for blk in vit.blocks:\n            a = getattr(blk, \'attn\', None)\n            if a is not None and hasattr(a, \'num_prefix_tokens\'):\n                a.num_prefix_tokens = a.num_prefix_tokens + 1\n\n    @staticmethod\n    def _maybe(mod, x):\n        return x if mod is None else mod(x)\n\n    def forward_features(self, x, cat):\n        v = self.vit\n        x = v.patch_embed(x)\n        pos = v._pos_embed(x)\n        rope = None\n        if isinstance(pos, tuple):\n            x, rope = pos\n        else:\n            x = pos\n        x = self._maybe(getattr(v, \'patch_drop\', None), x)\n        x = self._maybe(getattr(v, \'norm_pre\', None), x)\n        npt = self._orig_prefix\n        tok = self.tok(cat).unsqueeze(1)\n        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)\n        if rope is not None:\n            if getattr(v, \'rope_mixed\', False):\n                for i, blk in enumerate(v.blocks):\n                    x = blk(x, rope=rope[i])\n            else:\n                for blk in v.blocks:\n                    x = blk(x, rope=rope)\n        else:\n            x = v.blocks(x)\n        return v.norm(x)\n\n    def forward_head(self, x, pre_logits=True):\n        return self.vit.forward_head(x, pre_logits=pre_logits)\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\n\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\nclass _GatedDepthBlock(nn.Module):\n\n    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):\n        super().__init__()\n        self.norm = nn.GroupNorm(1, n_slice)\n        self.v = nn.Conv2d(n_slice, n_slice, 1)\n        self.g = nn.Conv2d(n_slice, n_slice, 1)\n        self.out = nn.Conv2d(n_slice, n_slice, 1)\n        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))\n        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()\n\n    def forward(self, x):\n        z = self.norm(x)\n        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))\n\nclass DepthCompress(nn.Module):\n\n    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):\n        super().__init__()\n        self.imagenet = imagenet\n        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])\n        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)\n        if imagenet:\n            self.register_buffer(\'mu\', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))\n            self.register_buffer(\'sd\', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))\n\n    def forward(self, x):\n        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)\n        z = x\n        for b in self.blocks:\n            z = b(z)\n        z = self.proj(z)\n        if self.imagenet:\n            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)\n        return z * keep\n\nN_PLANE, N_CONTRAST = (3, 2)\n\n_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2\n\n_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2\n\nclass SlotDepthMixer(nn.Module):\n\n    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):\n        super().__init__()\n        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)\n        self.alpha_max = alpha_max\n        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])\n        self.register_buffer(\'base\', b.log()[self.r:])\n        n_u = self.r + 1\n        self.shared = nn.Parameter(torch.zeros(n_u))\n        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))\n        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))\n        self.g0 = nn.Parameter(torch.zeros(()))\n        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))\n        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))\n        idx = torch.arange(n_slice)\n        self.register_buffer(\'off\', idx[None, :] - idx[:, None])\n\n    def kernel(self, slot):\n        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))\n        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]\n        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)\n        return F.softmax(full, dim=-1)\n\n    def alpha(self, slot):\n        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))\n        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])\n\n    def forward(self, x, slot, vmask):\n        T, S, H, W = x.shape\n        if vmask is None:\n            raise ValueError(\'stem=mixer requires the padding mask\')\n        k = self.kernel(slot)\n        v = vmask.to(k.dtype)\n        d = self.off + self.r\n        inb = (d >= 0) & (d < self.ksize)\n        kk = k[:, d.clamp(0, self.ksize - 1)] * inb\n        M = kk * v[:, None, :]\n        den = M.sum(-1, keepdim=True)\n        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)\n        ok = (den > 1e-06) & v[:, :, None].bool()\n        M = torch.where(ok, M / den.clamp(min=1e-06), eye)\n        a = self.alpha(slot)[:, None, None]\n        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)\n        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):\n            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))\n            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)\n        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)\n\ndef _seg_mean_max(v, sidx, B):\n    D = v.shape[1]\n    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))\n    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)\n    mean = mean / cnt.clamp(min=1).unsqueeze(1)\n    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)\n    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce=\'amax\', include_self=True)\n    return torch.cat([mean, mx], 1)\n\ndef _pad_kv(x, sidx, B, norm):\n    T, P, D = x.shape\n    cnt = torch.bincount(sidx, minlength=B)\n    S = int(cnt.max().item())\n    starts = torch.cumsum(cnt, 0) - cnt\n    pos = torch.arange(T, device=x.device) - starts[sidx]\n    pad = x.new_zeros(B, S, P, D)\n    pad[sidx, pos] = x\n    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)\n    keep[sidx, pos] = True\n    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))\n\nclass _GatedDelta(nn.Module):\n\n    def __init__(self, d, n_labels, n_heads, dropout):\n        super().__init__()\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.kv_norm = nn.LayerNorm(d)\n        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)\n        self.d_norm = nn.LayerNorm(d)\n        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))\n        self.db = nn.Parameter(torch.zeros(n_labels))\n        self.gate = nn.Parameter(torch.zeros(n_labels))\n\n    def delta(self, pat, sidx, B, return_attn):\n        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)\n        q = self.q.unsqueeze(0).expand(B, -1, -1)\n        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)\n        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)\n\nclass TokenResidualPool(_GatedDelta):\n\n    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):\n        super().__init__(d, n_labels, n_heads, dropout)\n        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))\n        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)\n        return (base + self.gate * d_, w)\n\nclass CodexResidualPool(_GatedDelta):\n\n    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):\n        super().__init__(d, n_labels, n_heads, dropout)\n        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))\n        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)\n        return (base + self.gate * d_, w)\n\nclass ClsAddPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):\n        super().__init__()\n        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)\n\nclass Readout(nn.Module):\n\n    def __init__(self, pool, d, n_labels=12, pe=64):\n        super().__init__()\n        self.pool_kind, self.k = (pool, n_labels)\n        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)\n        if pool in (\'xres\', \'clsadd\', \'xcodex\'):\n            self.pool = {\'xres\': TokenResidualPool, \'clsadd\': ClsAddPool, \'xcodex\': CodexResidualPool}[pool](d, n_labels, pe=pe)\n        elif pool in (\'attn\', \'xattn\'):\n            if pool == \'xattn\':\n                self.pool = TokenXAttnPool(d, n_labels)\n                wd = 3 * d + pe\n            else:\n                self.pool = LabelAttentionPool(d, n_labels)\n                wd = d + pe\n            self.norm = nn.LayerNorm(wd)\n            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))\n            self.b = nn.Parameter(torch.zeros(n_labels))\n        else:\n            self.pool = MeanMaxPool()\n            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))\n        self.drop = nn.Dropout(0.2)\n\n    def forward(self, f, slot, sidx, B, return_attn=False):\n        pe = self.pres_emb(slot)\n        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)\n        if self.pool_kind in (\'xres\', \'clsadd\', \'xcodex\'):\n            return self.pool(f, slot, sidx, B, pres)[0]\n        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)\n        if self.pool_kind in (\'attn\', \'xattn\'):\n            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)\n            x = self.drop(self.norm(x))\n            return (x * self.w).sum(-1) + self.b\n        return self.net(torch.cat([pooled, pres], 1))\n\nclass Net(nn.Module):\n\n    def __init__(self, enc, cond, n_meta=0, pool=\'mean_max\', stem=\'native\', n_slice=16):\n        super().__init__()\n        self.enc, self.cond = (enc, cond)\n        self.compress = DepthCompress(n_slice, 3) if stem == \'compress\' else None\n        self.mixer = SlotDepthMixer(n_slice) if stem == \'mixer\' else None\n        self.tokens = pool in (\'xattn\', \'xres\', \'clsadd\', \'xcodex\')\n        D = enc.num_features\n        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None\n        self.readout = Readout(pool, D)\n        if cond == \'post\':\n            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)\n\n    def forward(self, im, slot, smeta, sidx, B, vm=None):\n        if self.mixer is not None:\n            im = self.mixer(im, slot, vm)\n        if self.compress is not None:\n            im = self.compress(im)\n        f = self.enc.forward_features(im, slot) if self.cond == \'token\' else self.enc.forward_features(im)\n        if self.tokens:\n            inner = getattr(self.enc, \'vit\', self.enc)\n            orig = getattr(self.enc, \'_orig_prefix\', getattr(inner, \'num_prefix_tokens\', 1))\n            f = torch.cat([f[:, :1], f[:, orig:]], 1)\n        else:\n            f = self.enc.forward_head(f, pre_logits=True)\n            if f.dim() > 2:\n                f = f.flatten(1)\n        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v\n        if self.cond == \'post\':\n            f = f + ex(self.slot_emb(slot))\n        if self.meta_mlp is not None and smeta.shape[1] > 0:\n            mt = self.meta_mlp(smeta)\n            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt\n        return self.readout(f, slot, sidx, B)\n\ndef amp_for(dev):\n    if not str(dev).startswith(\'cuda\'):\n        return (torch.float32, False)\n    cc = torch.cuda.get_device_capability(dev)\n    if AMP_PREF == \'bf16\':\n        return (torch.bfloat16, True)\n    if AMP_PREF == \'fp16\':\n        return (torch.float16, True)\n    if AMP_PREF == \'fp32\':\n        return (torch.float32, False)\n    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)\n\nMICRO = 8\n\ndef _norm_(im):\n    k = CFG.get(\'norm\', \'none\')\n    if k == \'zscore\':\n        m = (im > 0).float()\n        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)\n        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n\n        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n\n        return (im - mu) / (var.sqrt() + 1e-06) * m\n    if k == \'imagenet\':\n        m = (im > 0).float()\n        return (im - 0.485) / 0.229 * m\n    return im\n\n@torch.no_grad()\ndef _micro(images, masks):\n    dev = DEV\n    ims, slots, sidx, vms = ([], [], [], [])\n    for b in range(len(masks)):\n        present = np.nonzero(masks[b] > 0)[0]\n        if len(present) == 0:\n            continue\n        blk = images[b][present]\n        ims.append(torch.from_numpy(blk))\n        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))\n        slots.append(torch.from_numpy(present + 1).long())\n        sidx.append(torch.full((len(present),), b, dtype=torch.long))\n    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)\n    if not ims:\n        return out\n    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))\n    sl = torch.cat(slots).to(dev)\n    si = torch.cat(sidx).to(dev)\n    vm = torch.cat(vms).to(dev)\n    sm = torch.zeros(len(sl), CFG.get(\'n_meta\', 0), device=dev)\n    per = torch.zeros(len(models), len(masks), len(LABELS), device=dev, dtype=torch.float32)\n    with torch.autocast(\'cuda\' if str(dev).startswith(\'cuda\') else \'cpu\', dtype=AMP_DT, enabled=AMP_ON):\n        for fold_index, model in enumerate(models):\n            per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(masks), vm=vm).float())\n    got = per.cpu().numpy()\n    keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])\n    out[:, keep] = got[:, keep]\n    return out\n\ndef predict(images, masks):\n    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)\n    for a in range(0, len(masks), MICRO):\n        b = min(a + MICRO, len(masks))\n        out[:, a:b] = _micro(images[a:b], masks[a:b])\n    return out\n'


In [ ]:
# 新しいOutputディレクトリへ保存。Kaggle上でだけ実行します。
REPORT = probe.run_probe(CONTRACT, DEFINITION_SOURCE, studies=64, seed=20261005, repeats=3)
print('診断結果:', REPORT['status'])
print('保存先:', REPORT['run_directory'])
print('採用判断:', REPORT['adoption_decision'])
